# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** CPU-only notebook authored and statically validated (four-method lamina reconstruction of embedded sample leaves raw/0001+0002 with pinned author code 4aabe9f25c0550b885c47cf4bb8d94f07c5591). Three supervised colab exec attempts on two fresh CPU runtimes all failed at the CLI's initial execute_code with WebSocketConnectionClosedException (x3) before any notebook cell ran; 0 saved output notebooks. Colab allocation was also severely degraded (assign long-polls of 3.5-13 min). EXECUTION FAILED / NOT VALIDATED; complete unexecuted notebook preserved in deliverables/ for the unverified-reference path. Not a missing-material conclusion.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Poplar-Studio: skeleton-guided reconstruction of editable 3D poplar seedling models from multi-view point clouds

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-cd2cf156846799a5a246f2cd226e37c7)


# Poplar-Studio: skeleton-guided 3D poplar seedling reconstruction — four-method lamina reconstruction demo

Reproduction of: **Z. Wang et al., "Poplar-Studio: skeleton-guided reconstruction of editable 3D poplar seedling models from multi-view point clouds", *Computers and Electronics in Agriculture* 256:112454 (2026)**, DOI: [10.1016/j.compag.2026.112454](https://doi.org/10.1016/j.compag.2026.112454)

Author code: `wanghanruibaba/Poplar-Studio` @ commit `4aabe9f25c05` (checked inside this notebook).

**Scope (partial demonstration).** The paper converts multi-view point clouds of poplar seedlings into editable organ-level 3D models. The paper's core leaf-area validation compares four lamina-reconstruction methods — the proposed **PCA boundary-constrained meshing** plus **adaptive Poisson**, **fixed Poisson**, and **ball pivoting** baselines — against independently measured leaf areas (paper-level: R² = 0.943, relative RMSE 10.3% over a 240-leaf benchmark). This notebook re-runs **all four methods on the two embedded sample leaves (`raw/0001/0001`, `raw/0002/0002`)** using the author's original, unmodified code, and compares the reconstructed areas against the author-frozen benchmark table (`benchmark/sampled_four_method_leaf_areas.csv`) and the released benchmark meshes.

**Not covered:** SfM-MVS image acquisition, PointNet++ leaf/stem segmentation, skeleton graph construction, stem validation, and the full 240-leaf benchmark are out of scope. A two-leaf run is a plausibility check, not verification of published dataset-level accuracy.

**実行範囲(日本語):** 埋め込みサンプル葉2枚(raw/0001, raw/0002)に対し、著者コードをそのまま用いてPCA法+3つのベースライン法の葉面積再構成を実行し、著者公開のベンチマーク表・メッシュと面積を比較します。SfM・点群セグメンテーション・骨格解析・240葉ベンチマーク全体は対象外です。

## Runtime: CPU only

The author's reconstruction code uses only NumPy, SciPy, and Open3D **on CPU** (no torch, no CUDA operations anywhere in the pinned repository). Poisson reconstruction depth ≤ 11 on a single leaf cloud (a few thousand points) takes seconds on CPU, so **no GPU is needed**: select **Runtime → Change runtime type → CPU** (the default free runtime).

Expected duration: ~5 minutes total (≈2 min Open3D install + ≈1 min download + ≈1–2 min reconstruction). Downloads: ≈15 MB.

**実行環境(日本語):** 著者コードはNumPy/SciPy/Open3DのCPU実装のみで、GPUは不要です。ランタイム種別はCPUのままで実行してください。所要時間は全体で約5分、ダウンロード約15MBです。

### Check the environment

We print the Python, NumPy and SciPy versions and the CPU resources of the allocated runtime, so the recorded result is tied to actual observed hardware. No GPU is requested or used.

**実行内容:** 割り当てられたCPUランタイムのPython/NumPy/SciPyバージョンとCPU資源を確認します。GPUは使用しません。

In [ ]:
import platform, os, multiprocessing
import numpy as np, scipy
print("Python:", platform.python_version())
print("numpy:", np.__version__, "| scipy:", scipy.__version__)
print("CPU cores:", multiprocessing.cpu_count())
mem_gb = None
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                mem_gb = int(line.split()[1]) / 1024**2
                print(f"Total RAM: {mem_gb:.1f} GiB")
                break
except OSError as e:
    print("meminfo unavailable:", e)

### Install Open3D

`pca_reconstruct.py` (the proposed PCA method) needs only NumPy/SciPy, but `baseline_core.py` imports **Open3D** for Poisson reconstruction, ball pivoting, mesh cleanup, and density cropping. We pin `open3d==0.19.0`, the release providing stable Python 3.12 manylinux wheels for the Colab CPU image.

**実行内容:** ベースライン法(open3d利用)のためOpen3D 0.19.0をインストールします。

In [ ]:
%pip install -q open3d==0.19.0
import open3d as o3d
print("open3d:", o3d.__version__)

### Download the pinned author repository (partial clone, exact commit)

Inside this Colab VM (never on the authoring host), we make a **blobless partial clone** and check out the exact verified commit `4aabe9f25c0550b885c47cf4bb8d94f07c559100` in detached HEAD, with a sparse checkout limited to what this demo needs: the four author code files, `leaf_manifest.csv`, the embedded sample leaves `raw/0001` + `raw/0002`, and the `benchmark/` folder (archived per-method meshes + reference CSVs). The checkout is asserted to match the pinned commit.

**実行内容:** 著者リポジトリを部分クローンし、検証済みコミット4aabe9f25c05をスパースチェックアウトします。

In [ ]:
import subprocess, os
REPO = "/content/Poplar-Studio"
COMMIT = "4aabe9f25c0550b885c47cf4bb8d94f07c559100"
URL = "https://github.com/wanghanruibaba/Poplar-Studio.git"
FILES = ["leaf_manifest.csv", "pca_reconstruct.py", "baseline_core.py",
         "reconstruct.py", "recalculate_areas.py", "raw/0001", "raw/0002", "benchmark"]

def sh(*args, **kw):
    return subprocess.run(args, check=True, capture_output=True, text=True, **kw)

if not os.path.isdir(os.path.join(REPO, ".git")):
    sh("git", "init", "-q", REPO)
    sh("git", "-C", REPO, "remote", "add", "origin", URL)
    sh("git", "-C", REPO, "config", "extensions.partialClone", "origin")
    sh("git", "-C", REPO, "sparse-checkout", "init", "--no-cone")
    sh("git", "-C", REPO, "sparse-checkout", "set", *FILES)
sh("git", "-C", REPO, "fetch", "--depth=1", "--filter=blob:none", "origin", COMMIT)
sh("git", "-C", REPO, "checkout", "-q", "FETCH_HEAD")
head = sh("git", "-C", REPO, "rev-parse", "HEAD").stdout.strip()
assert head == COMMIT, f"checkout {head} != pinned {COMMIT}"
print("Checked out pinned commit:", head)

### Verify the retrieved files

We list the checked-out sample data and code with byte sizes, and assert the two demo leaves and their paired author PCA meshes exist. Sizes are cross-checked against the pinned GitHub tree metadata (raw/ = 84 files ≈ 13.7 MB across `0001`+`0002`; benchmark CSVs ≈ 75 KB).

**実行内容:** 取得したサンプル点群・コード・ベンチマークファイルの実在とサイズを確認します。

In [ ]:
import os
REPO = "/content/Poplar-Studio"
for name in ["leaf_manifest.csv", "pca_reconstruct.py", "baseline_core.py",
             "recalculate_areas.py", "reconstruct.py"]:
    p = os.path.join(REPO, name)
    assert os.path.isfile(p), p
    print(f"{name}: {os.path.getsize(p):,} bytes")
for d in ["raw/0001", "raw/0002", "benchmark"]:
    paths = []
    for root, _, files in os.walk(os.path.join(REPO, d)):
        paths += [os.path.join(root, f) for f in files]
    print(f"{d}: {len(paths)} files, {sum(os.path.getsize(p) for p in paths):,} bytes")
for leaf in ["0001", "0002"]:
    for ext in [".txt", "_pca_mesh.obj"]:
        p = os.path.join(REPO, "raw", leaf, leaf + ext)
        assert os.path.isfile(p), p
print("Sample leaves 0001 and 0002 present with paired author PCA meshes.")

### Load the frozen manifest and the benchmark area table

`leaf_manifest.csv` maps each leaf ID to genotype, physical `scale_factor` (point units → cm), point-cloud path, author PCA-mesh path, benchmark flag, and the paper's PCA `pca_area_cm2`. `benchmark/sampled_four_method_leaf_areas.csv` holds the author-frozen four-method areas (`<method>_area_cm2`) plus the independent `measured_area_cm2` reference. We display the rows of the two demo leaves.

**実行内容:** 葉マニフェスト(遺伝型・スケール係数・PCA面積)とベンチマーク面積表を読み込み、デモ対象2葉の行を表示します。

In [ ]:
import pandas as pd
REPO = "/content/Poplar-Studio"
manifest = pd.read_csv(os.path.join(REPO, "leaf_manifest.csv"), encoding="utf-8-sig")
benchmark = pd.read_csv(os.path.join(REPO, "benchmark", "sampled_four_method_leaf_areas.csv"),
                        encoding="utf-8-sig")
print("manifest columns:", list(manifest.columns))
print("manifest rows:", len(manifest))
print("benchmark columns:", list(benchmark.columns))
print("benchmark rows:", len(benchmark))
demo = ["0001", "0002"]
print("\n--- manifest rows for demo leaves ---")
print(manifest[manifest["leaf"].isin(demo)].to_string(index=False))
print("\n--- benchmark rows for demo leaves ---")
print(benchmark[benchmark["leaf"].isin(demo)].to_string(index=False))

### Preview the sample input and the author's PCA reconstruction (leaf 0001)

Before any reconstruction we visualize the actual data: left, the raw per-leaf point cloud `raw/0001/0001.txt` (XY view, colored by height); right, the vertex set of the author's released PCA reconstruction `raw/0001/0001_pca_mesh.obj` for the same leaf. This is the author-provided reference reconstruction shipped with the sample — not our result.

**実行内容:** デモ葉0001の生点群(XY投影、高さ着色)と、著者公開のPCA再構成メッシュの頂点を並べて表示します。右は著者提供の参照出力であり本ノートブックの結果ではありません。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
REPO = "/content/Poplar-Studio"

def load_xyz(path):
    try:
        d = np.loadtxt(path)
    except ValueError:
        d = np.loadtxt(path, delimiter=",")
    return d[:, :3][np.all(np.isfinite(d[:, :3]), axis=1)]

def load_obj_vertices(path):
    vs = []
    with open(path) as f:
        for line in f:
            if line.startswith("v "):
                vs.append([float(x) for x in line.split()[1:4]])
    return np.asarray(vs)

pts = load_xyz(f"{REPO}/raw/0001/0001.txt")
mesh_v = load_obj_vertices(f"{REPO}/raw/0001/0001_pca_mesh.obj")
print(f"leaf 0001: {len(pts)} input points | author PCA mesh: {len(mesh_v)} vertices")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
s1 = axes[0].scatter(pts[:, 0], pts[:, 1], c=pts[:, 2], s=1.5, cmap="viridis")
axes[0].set_title("Input point cloud raw/0001/0001.txt (XY, color = Z)")
axes[0].set_aspect("equal"); fig.colorbar(s1, ax=axes[0], label="Z")
axes[1].scatter(mesh_v[:, 0], mesh_v[:, 1], c=mesh_v[:, 2], s=1.5, cmap="viridis")
axes[1].set_title("Author PCA mesh 0001_pca_mesh.obj (XY, color = Z) [reference]")
axes[1].set_aspect("equal")
plt.tight_layout(); plt.savefig("/content/poplar_preview_0001.png", dpi=150); plt.show()

### Run the proposed PCA lamina reconstruction (leaves 0001, 0002)

We call the author's `pca_reconstruct.process_file` with the paper's frozen defaults (`ReconstructionConfig()`: min 30 points, ≤5000 vertices, alpha factor 2.5, MLS height smoothing k=18, blend 0.65 — identical to the defaults used by the author's frozen-cohort driver `reconstruct.py`). It writes `<leaf>_pca_mesh.obj` (open single-layer surface) plus PLY/JSON to `/content/runs/reconstruction/pca/`, and reports mesh size, boundary loops, and internal holes.

**実行内容:** 著者の`process_file`を既定パラメータのまま2葉に実行し、PCA法の開放メッシュを生成します。

In [ ]:
import sys, time
sys.path.insert(0, REPO)
from pathlib import Path
import pca_reconstruct

out_dir = Path("/content/runs/reconstruction/pca"); out_dir.mkdir(parents=True, exist_ok=True)
pca_records = {}
for leaf in ["0001", "0002"]:
    t0 = time.perf_counter()
    rec = pca_reconstruct.process_file(Path(f"{REPO}/raw/{leaf}/{leaf}.txt"), out_dir,
                                       pca_reconstruct.ReconstructionConfig())
    assert rec.get("success"), rec
    rec["seconds"] = time.perf_counter() - t0
    pca_records[leaf] = rec
    print(f"leaf {leaf}: {rec['mesh_vertices']} vertices, {rec['mesh_triangles']} triangles, "
          f"internal_holes={rec['internal_holes']}, {rec['seconds']:.2f} s")

### Run the three baseline reconstructions (adaptive Poisson, fixed Poisson, ball pivoting)

We call the author's `baseline_core.process_one(path, out_dir, method, 'reduced')` — the same entry point and `"reduced"` search budget used by the author's `reconstruct.py`. Adaptive Poisson searches a small candidate set (target-point multipliers 0.8/1.0/1.2, ≤2 refinement iterations) seeded with `RNG_SEED = 20240101`; fixed Poisson uses the frozen constants (3000 target points, depth 9); ball pivoting uses radii = median spacing × {1, 2, 4}. Meshes are written as `<leaf>_<method>.obj`. We also call the author's `_init_worker()` to match the serial single-worker execution of the frozen run.

**実行内容:** 著者の`process_one`を`reduced`予算で3ベースライン×2葉に実行します(シード20240101)。

In [ ]:
import baseline_core

baseline_core._init_worker()
BASE_OUT = {}
for method in ["adaptive_poisson", "fixed_poisson", "ball_pivoting"]:
    out_dir = Path(f"/content/runs/reconstruction/{method}"); out_dir.mkdir(parents=True, exist_ok=True)
    for leaf in ["0001", "0002"]:
        t0 = time.perf_counter()
        rec = baseline_core.process_one(f"{REPO}/raw/{leaf}/{leaf}.txt", str(out_dir),
                                        method, "reduced")
        assert rec.get("success"), rec
        rec["wall_seconds"] = time.perf_counter() - t0
        BASE_OUT[(leaf, method)] = rec
        print(f"leaf {leaf} {method}: {rec['mesh_vertices']} verts, "
              f"{rec['mesh_triangles']} tris, area={rec['area']:.3f}, {rec['wall_seconds']:.1f} s")

### Compare reconstructed areas with the author-frozen benchmark

Using the author's exact area routine (`recalculate_areas.surface_area`, triangle cross-product sum), we compute cm² areas (`raw area × scale_factor²`) for (a) our four fresh reconstructions and (b) the author's released benchmark meshes in `benchmark/<method>/`, then compare everything with the frozen table's `<method>_area_cm2` (validates our area pipeline against author-frozen numbers — we print the maximum absolute difference, like the author's `recalculate_areas.main`) and with the independent `measured_area_cm2` reference.

**実行内容:** 著者の面積計算ルーチンで、本ノートブックの再構成結果と著者公開ベンチマークメッシュの両方の面積をcm²で算出し、凍結表および独立実測値と比較します。

In [ ]:
from recalculate_areas import surface_area
import pandas as pd

scale = {r["leaf"]: float(r["scale_factor"]) for _, r in
         manifest[manifest["leaf"].isin(["0001", "0002"])].iterrows()}
METHODS = ["pca", "adaptive_poisson", "fixed_poisson", "ball_pivoting"]

def obj_path(method, leaf, fresh):
    name = f"{leaf}_pca_mesh.obj" if method == "pca" else f"{leaf}_{method}.obj"
    return (f"/content/runs/reconstruction/{method}/{name}" if fresh
            else f"{REPO}/benchmark/{method}/{name}")

rows = []
max_frozen_diff = 0.0
for method in METHODS:
    for leaf in ["0001", "0002"]:
        fresh = surface_area(obj_path(method, leaf, True)) * scale[leaf] ** 2
        released = surface_area(obj_path(method, leaf, False)) * scale[leaf] ** 2
        brow = benchmark[benchmark["leaf"] == leaf].iloc[0]
        archived = float(brow[f"{method}_area_cm2"])
        max_frozen_diff = max(max_frozen_diff, abs(released - archived))
        rows.append({"leaf": leaf, "method": method, "scale_factor": scale[leaf],
                     "fresh_area_cm2": round(fresh, 4),
                     "released_benchmark_area_cm2": round(released, 4),
                     "archived_area_cm2": archived,
                     "measured_area_cm2": float(brow["measured_area_cm2"]),
                     "fresh_vs_archived_pct": round(100 * (fresh - archived) / archived, 3),
                     "fresh_vs_measured_pct": round(100 * (fresh - brow["measured_area_cm2"])
                                                    / brow["measured_area_cm2"], 3)})
print(f"Max |released benchmark mesh area - archived table area| = {max_frozen_diff:.6g} cm2"
      "  (area-pipeline check against author-frozen numbers)")
results = pd.DataFrame(rows)
display(results)

### Visualize the four-method area comparison

For each demo leaf we plot the four methods' **fresh areas from this notebook's run** against the **author-frozen archived areas**, with the independent measured reference as a dashed line. This graph is generated here from the actual executed results (an additional visualization created for this notebook, not a figure reproduced from the paper).

**実行内容:** 本ノートブックの実行結果と著者凍結値・独立実測値を棒グラフで比較します(本ノートブック独自の追加可視化であり、論文図の再現ではありません)。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
x = np.arange(len(METHODS)); w = 0.38
for ax, leaf in zip(axes, ["0001", "0002"]):
    sub = results[results["leaf"] == leaf].set_index("method").loc[METHODS]
    ax.bar(x - w / 2, sub["fresh_area_cm2"], w, label="fresh (this notebook)")
    ax.bar(x + w / 2, sub["archived_area_cm2"], w, label="archived (author benchmark)",
           hatch="//", alpha=0.8)
    ax.axhline(sub["measured_area_cm2"].iloc[0], ls="--", c="k", lw=1,
               label="independent measured reference")
    ax.set_xticks(x, METHODS, rotation=20)
    ax.set_ylabel("leaf area (cm²)")
    ax.set_title(f"leaf {leaf}")
    ax.legend(fontsize=8)
plt.suptitle("Four-method lamina reconstruction areas: this run vs author-frozen benchmark")
plt.tight_layout(); plt.savefig("/content/poplar_area_comparison.png", dpi=150); plt.show()

### Interpretation and limitations

- **What is verified here:** the author's original code (pinned commit) runs end-to-end in Colab on the embedded sample leaves; our recomputed areas of the *released* benchmark meshes match the frozen table, and our *fresh* reconstructions can be compared with those frozen values per method.
- **What is not verified:** the paper's dataset-level statistics (R² = 0.943, relative RMSE 10.3% over 240 leaves), the SfM/segmentation/skeleton stages, and any claim about the full pipeline. Two leaves is a plausibility demonstration.
- Numerical variation: Poisson and ball-pivoting contain seeded resampling (`RNG_SEED = 20240101`); small area differences vs the frozen values are expected across platforms/Open3D versions.

**結果の解釈(日本語):** 本ノートブックは著者コードがColab上で完走し、公開ベンチマークメッシュの面積が凍結表と一致すること、また新規再構成の面積を凍結値と比較できることを示します。240葉の統計量やSfM・セグメンテーション・骨段階は検証対象外です。Poisson法・BPA法はシード付きリサンプリングを含むため、環境差によるわずかな面積差は想定内です。

### Export the comparison table

We save the executed comparison table as CSV in the Colab VM (`/content/poplar_area_results.csv`) so the learner can download it.

**実行内容:** 比較表をCSVとして保存します。

In [ ]:
results.to_csv("/content/poplar_area_results.csv", index=False, encoding="utf-8-sig")
print("saved /content/poplar_area_results.csv")
print(results.to_string(index=False))

## References

- Paper: *Computers and Electronics in Agriculture* 256:112454 (2026), DOI 10.1016/j.compag.2026.112454 (published ScienceDirect HTML, online 2026-09-22, CC BY-NC-ND).
- Code: https://github.com/wanghanruibaba/Poplar-Studio @ commit `4aabe9f25c0550b885c47cf4bb8d94f07c559100` — `pca_reconstruct.py` (PCA method), `baseline_core.py` (baselines), `recalculate_areas.py` (area routine), `reconstruct.py` (frozen-cohort driver, mirrored here).
- Data: embedded sample leaves `raw/0001`, `raw/0002` and `benchmark/` meshes + `sampled_four_method_leaf_areas.csv` from the same pinned commit.
- PhenoPaper investigation `p-cd2cf156846799a5a246f2cd226e37c7-20261019T110852Z-645682` was used as research guidance; all scientific steps above were verified against the pinned author code directly.

**参考(日本語):** 上記の論文・コミット固定の著者コード・埋め込みサンプルデータのみに基づいています。